In [1]:
!pip install google-cloud-bigquery pandas scikit-learn xgboost --quiet

In [12]:
from google.cloud import bigquery
import pandas as pd

# connect to BigQuery
client = bigquery.Client(project='no-ssg-gcp-miden-isnd')

# pull training data
query = """
SELECT
    season,
    team,
    is_prediction_row,
    current_points,
    points_per_game,
    avg_goals_scored,
    avg_goals_conceded,
    goal_difference / matches_played        as gd_per_game,
    home_points / matches_played            as home_ppg,
    away_points / matches_played            as away_ppg,
    avg_shots_on_target,
    avg_shots_on_target_against,
    shot_accuracy,
    last_5_ppg,
    last_10_ppg,
    avg_implied_win_prob,
    matches_played,
    final_points
FROM `no-ssg-gcp-miden-isnd.analytics_epl.fct_training_spine`
"""

df = client.query(query).to_dataframe()
print(df.shape)
print(df.head())

(220, 18)
    season        team  is_prediction_row  current_points  points_per_game  \
0  2015/16   Leicester              False              66            2.129   
1  2015/16   Tottenham              False              61            1.968   
2  2015/16     Arsenal              False              58            1.871   
3  2015/16    Man City              False              54            1.742   
4  2015/16  Man United              False              53            1.710   

   avg_goals_scored  avg_goals_conceded  gd_per_game  home_ppg  away_ppg  \
0             1.742               1.000     0.741935  1.032258  1.096774   
1             1.806               0.774     1.032258  1.032258  0.935484   
2             1.677               0.968     0.709677  0.967742  0.903226   
3             1.806               1.032     0.774194  1.000000  0.741935   
4             1.258               0.871     0.387097  1.000000  0.709677   

   avg_shots_on_target  avg_shots_on_target_against  shot_accura

In [13]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# define features
features = [
    'points_per_game',
    'avg_goals_scored',
    'avg_goals_conceded',
    'gd_per_game',
    'home_ppg',
    'away_ppg',
    'avg_shots_on_target',
    'avg_shots_on_target_against',
    'shot_accuracy',
    'last_5_ppg',
    'last_10_ppg',
    'avg_implied_win_prob',
    'matches_played',
    'current_points'
]

# split into training and prediction sets
train = df[df['is_prediction_row'] == False].copy()
predict = df[df['is_prediction_row'] == True].copy()

# fill nulls in avg_implied_win_prob with median
median_prob = train['avg_implied_win_prob'].median()
train['avg_implied_win_prob'] = train['avg_implied_win_prob'].fillna(median_prob)
predict['avg_implied_win_prob'] = predict['avg_implied_win_prob'].fillna(median_prob)

# training data
X_train = train[features]
y_train = train['final_points']

# train xgboost model
model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# evaluate on training data
y_pred_train = model.predict(X_train)
mae = mean_absolute_error(y_train, y_pred_train)
r2 = r2_score(y_train, y_pred_train)

print(f"MAE: {mae:.3f}")
print(f"R²:  {r2:.3f}")

MAE: 1.239
R²:  0.992


In [14]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np

# define features
features = [
    'points_per_game',
    'avg_goals_scored',
    'avg_goals_conceded',
    'gd_per_game',
    'home_ppg',
    'away_ppg',
    'avg_shots_on_target',
    'avg_shots_on_target_against',
    'shot_accuracy',
    'last_5_ppg',
    'last_10_ppg',
    'avg_implied_win_prob',
    'matches_played',
    'current_points'
]

# split into training and prediction sets
train = df[df['is_prediction_row'] == False].copy()
predict = df[df['is_prediction_row'] == True].copy()

# fill nulls
median_prob = train['avg_implied_win_prob'].median()
train['avg_implied_win_prob'] = train['avg_implied_win_prob'].fillna(median_prob)
predict['avg_implied_win_prob'] = predict['avg_implied_win_prob'].fillna(median_prob)

X = train[features]
y = train['final_points']

# split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

# train xgboost model
model2 = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model2.fit(X_train, y_train)

# evaluate on HELD OUT test set
y_pred_test = model2.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"\nTest MAE: {mae:.3f}")
print(f"Test R²:  {r2:.3f}")

Training rows: 160
Test rows:     40

Test MAE: 3.606
Test R²:  0.902


In [15]:
# generate predictions on 2025/26
X_predict = predict[features]
predicted_points = model.predict(X_predict)

# build results dataframe
results = predict[['team', 'current_points']].copy()
results['xgboost_prediction'] = predicted_points.round()
results['bqml_prediction'] = [84, 77, 67, 64, 61, 59, 56, 55, 53, 53, 52, 52, 51, 48, 43, 40, 39, 34, 25, 23]
results = results.sort_values('xgboost_prediction', ascending=False).reset_index(drop=True)
results.index += 1

print(results.to_string())


              team  current_points  xgboost_prediction  bqml_prediction
1          Arsenal              70                85.0               84
2         Man City              61                74.0               77
3       Man United              55                66.0               67
4      Aston Villa              54                66.0               64
5          Chelsea              48                62.0               59
6        Liverpool              49                61.0               61
7        Brentford              46                56.0               56
8          Everton              46                56.0               55
9         Brighton              43                54.0               52
10       Newcastle              42                51.0               52
11  Crystal Palace              39                51.0               48
12          Fulham              44                50.0               53
13      Sunderland              43                49.0          